# Stage 4 AlignTune Training & Experiment Exploratory Analysis (`E1`–`E4B`)

Comprehensive analysis of all five merged LoRA post-training experiments on **`CohereLabs/tiny-aya-global`** (`r=16`, `alpha=32`, `effective_batch_size=16`, `completion_only_loss=True`, `lr_scheduler_type="cosine_with_restarts"`, `warmup_ratio=0.10`, `max_epochs=10`), combining:
- TensorBoard `.tfevents` training & validation scalars (`models/adapters/*/runs/`)
- Adapter execution metadata (`training_summary.json`, `training_started.json`, `lexsi_provenance.json`)
- Stage 2.5 SFT dataset compositions (`data/sft/*`) vs held-out PhysGym splits (`data/splits/*`)

| Run | Role | Adapter Directory | Dataset Rows | Unique Problems | Peak LR | Validation Source | Saved `final_adapter/` |
| :--- | :--- | :--- | ---: | ---: | :---: | :--- | :---: |
| **`E1`** | Solver | `solver_e1_manual_det_lora` | 1,076 | 1,076 | `2.0e-4` | `data/splits/validation.jsonl` (38 raw PhysGym) | No *(crashed on save; Ep2 best)* |
| **`E2`** | Solver | `solver_e2_curatorkit_nollm_lora` | 728 | 728 | `2.0e-4` | `data/splits/validation.jsonl` (38 raw PhysGym) | Yes *(5 epochs, 230 steps)* |
| **`E3`** | Solver | `solver_e3_canonical_llm_lora` | 681 | 681 | `2.0e-4` | `data/splits/validation.jsonl` (38 raw PhysGym) | Yes *(4 epochs, 172 steps)* |
| **`E4A`** | Solver | `solver_e4_adversarial_lora` | 1,444 | 681 | `2.0e-4` | `data/splits/validation.jsonl` (38 raw PhysGym) | Yes *(4 epochs, 364 steps)* |
| **`E4B`** | Verifier | `verifier_e4_curated_lora` | 1,511 | 681 | `1.5e-4` | 10% row-random internal split (~151–152 rows) | Yes *(5 epochs, 425 steps)* |

### Key Questions Addressed
1. **Validation divergence:** Why did `eval/loss` rise after Epoch 1 for **`E3`** and **`E4A`**, while `eval/mean_token_accuracy` rose at Epoch 2 for **`E3`** (`0.6589 → 0.6622`) yet collapsed monotonically for **`E4A`** (`0.6611 → 0.6282`)?
2. **Exposure & step alignment:** How do **`E3`** and **`E4A`** align once normalized by optimizer steps and effective passes over the `681` unique problems?
3. **Training stability, entropy & LR shocks:** How do `train/grad_norm`, `train/entropy`, FLOPs, and throughput differ across raw textbook vs `aya-8b`-canonicalized runs — and what happens when `cosine_with_restarts` resets at Epoch 4.1 (`E2`, `E4B`)?
4. **Verifier & domain-shift audit (`E4B` + `Val/Test`):** What class imbalance, domain shift, and `problem_id` leakage risks exist in **`verifier_e4_curated`** and `data/splits/{validation,test}.jsonl`?

In [ ]:
from __future__ import annotations

import json
import re
import struct
import subprocess
from collections import Counter
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#fcfcfd",
    "axes.edgecolor": "#cbd5e1",
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--",
    "font.size": 10.5,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "legend.frameon": True,
    "legend.facecolor": "white",
    "legend.edgecolor": "#e2e8f0",
})


def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "configs" / "training" / "aligntune_training.yaml").is_file():
            return candidate
    return cur


def show_df(df: pd.DataFrame) -> None:
    if "display" in globals():
        display(df)  # type: ignore[name-defined]
    else:
        print(df.to_string(index=False))


REPO_ROOT = find_repo_root()
SNAPSHOT_PATH = REPO_ROOT / "experiments" / "data" / "tensorboard_scalars_snapshot.json"

EXPERIMENTS: list[dict[str, Any]] = [
    {
        "run": "E1",
        "label": "E1 (Manual Det, 1076)",
        "adapter": "solver_e1_manual_det_lora",
        "git_refs": ["HEAD", "13fa48a"],
        "role": "solver",
        "dataset_records": 1076,
        "unique_problems": 1076,
        "color": "#64748b",
        "marker": "o",
    },
    {
        "run": "E2",
        "label": "E2 (CuratorKIT No-LLM, 728)",
        "adapter": "solver_e2_curatorkit_nollm_lora",
        "git_refs": ["HEAD", "origin/run-e2"],
        "role": "solver",
        "dataset_records": 728,
        "unique_problems": 728,
        "color": "#0ea5e9",
        "marker": "s",
    },
    {
        "run": "E3",
        "label": "E3 (Canonical Aya-8B, 681)",
        "adapter": "solver_e3_canonical_llm_lora",
        "git_refs": ["HEAD", "origin/run-e3"],
        "role": "solver",
        "dataset_records": 681,
        "unique_problems": 681,
        "color": "#10b981",
        "marker": "^",
    },
    {
        "run": "E4A",
        "label": "E4A (Adversarial Solver, 1444)",
        "adapter": "solver_e4_adversarial_lora",
        "git_refs": ["HEAD", "origin/run-e4a"],
        "role": "solver",
        "dataset_records": 1444,
        "unique_problems": 681,
        "color": "#ef4444",
        "marker": "D",
    },
    {
        "run": "E4B",
        "label": "E4B (Adversarial Verifier, 1511)",
        "adapter": "verifier_e4_curated_lora",
        "git_refs": ["HEAD", "origin/run-e4b"],
        "role": "verifier",
        "dataset_records": 1511,
        "unique_problems": 681,
        "color": "#8b5cf6",
        "marker": "P",
    },
]
EXP_META = {e["run"]: e for e in EXPERIMENTS}
SOLVERS = [e for e in EXPERIMENTS if e["role"] == "solver"]


# --- Zero-dependency TFRecord + TensorBoard Event Protobuf Scalar Parser ---
def _read_varint(buf: bytes, pos: int) -> tuple[int, int]:
    res = 0
    shift = 0
    while pos < len(buf):
        b = buf[pos]
        pos += 1
        res |= (b & 0x7F) << shift
        if not (b & 0x80):
            return res, pos
        shift += 7
    return res, pos


def _skip_field(buf: bytes, pos: int, wire_type: int) -> int:
    if wire_type == 0:
        _, pos = _read_varint(buf, pos)
    elif wire_type == 1:
        pos += 8
    elif wire_type == 2:
        length, pos = _read_varint(buf, pos)
        pos += length
    elif wire_type == 5:
        pos += 4
    else:
        raise ValueError(f"Unsupported protobuf wire_type={wire_type}")
    return pos


def _parse_summary_value(buf: bytes) -> tuple[str, float | None]:
    pos = 0
    tag = ""
    val: float | None = None
    while pos < len(buf):
        key, pos = _read_varint(buf, pos)
        fn, wt = key >> 3, key & 7
        if fn == 1 and wt == 2:
            length, pos = _read_varint(buf, pos)
            tag = buf[pos : pos + length].decode("utf-8", errors="replace")
            pos += length
        elif fn == 2 and wt == 5:
            val = float(struct.unpack("<f", buf[pos : pos + 4])[0])
            pos += 4
        elif fn == 8 and wt == 2:
            length, pos = _read_varint(buf, pos)
            tbuf = buf[pos : pos + length]
            pos += length
            tp = 0
            while tp < len(tbuf):
                tk, tp = _read_varint(tbuf, tp)
                tfn, twt = tk >> 3, tk & 7
                if tfn == 4 and twt == 2:
                    tl, tp = _read_varint(tbuf, tp)
                    if tl == 4:
                        val = float(struct.unpack("<f", tbuf[tp : tp + 4])[0])
                    elif tl == 8:
                        val = float(struct.unpack("<d", tbuf[tp : tp + 8])[0])
                    tp += tl
                elif tfn == 5 and twt == 5:
                    val = float(struct.unpack("<f", tbuf[tp : tp + 4])[0])
                    tp += 4
                else:
                    tp = _skip_field(tbuf, tp, twt)
        else:
            pos = _skip_field(buf, pos, wt)
    return tag, val


def parse_tfevents_bytes(raw: bytes) -> list[dict[str, Any]]:
    offset = 0
    rows: list[dict[str, Any]] = []
    n = len(raw)
    while offset + 12 <= n:
        length = struct.unpack("<Q", raw[offset : offset + 8])[0]
        offset += 12  # 8-byte len + 4-byte masked CRC32C
        if offset + length + 4 > n:
            break
        ev = raw[offset : offset + length]
        offset += length + 4
        pos = 0
        step = 0
        wall_time = 0.0
        while pos < len(ev):
            key, pos = _read_varint(ev, pos)
            fn, wt = key >> 3, key & 7
            if fn == 1 and wt == 1:
                wall_time = float(struct.unpack("<d", ev[pos : pos + 8])[0])
                pos += 8
            elif fn == 2 and wt == 0:
                step, pos = _read_varint(ev, pos)
            elif fn == 5 and wt == 2:
                sl, pos = _read_varint(ev, pos)
                sbuf = ev[pos : pos + sl]
                pos += sl
                sp = 0
                while sp < len(sbuf):
                    sk, sp = _read_varint(sbuf, sp)
                    sfn, swt = sk >> 3, sk & 7
                    if sfn == 1 and swt == 2:
                        vl, sp = _read_varint(sbuf, sp)
                        tag, val = _parse_summary_value(sbuf[sp : sp + vl])
                        sp += vl
                        if tag and val is not None:
                            rows.append({
                                "step": int(step),
                                "tag": tag,
                                "value": float(val),
                                "wall_time": round(wall_time, 3),
                            })
                    else:
                        sp = _skip_field(sbuf, sp, swt)
            else:
                pos = _skip_field(ev, pos, wt)
    return rows


def _pick_best_completed_stream(
    candidates: list[tuple[str, list[dict[str, Any]]]]
) -> tuple[str, list[dict[str, Any]]] | None:
    """Prefer streams with completed evals; break ties by newest wall_time/path."""
    if not candidates:
        return None
    scored: list[tuple[int, float, str, list[dict[str, Any]]]] = []
    for name, events in candidates:
        eval_count = sum(1 for e in events if e["tag"] == "eval/loss")
        max_wt = max((e["wall_time"] for e in events), default=0.0)
        scored.append((1 if eval_count > 0 else 0, max_wt, name, events))
    best = max(scored, key=lambda x: (x[0], x[1], x[2]))
    return best[2], best[3]


def load_all_experiments(prefer_local: bool = True, prefer_git: bool = True) -> tuple[pd.DataFrame, dict[str, str], list[dict[str, Any]]]:
    snapshot: dict[str, Any] = {}
    if SNAPSHOT_PATH.is_file():
        snapshot = json.loads(SNAPSHOT_PATH.read_text(encoding="utf-8"))

    all_rows: list[dict[str, Any]] = []
    sources: dict[str, str] = {}
    stream_inventory: list[dict[str, Any]] = []

    for exp in EXPERIMENTS:
        run_id = exp["run"]
        chosen_events: list[dict[str, Any]] | None = None
        source_label = "missing"

        # 1. Local models/adapters/<adapter>/runs/
        if prefer_local:
            local_runs_dir = REPO_ROOT / "models" / "adapters" / exp["adapter"] / "runs"
            if local_runs_dir.is_dir():
                cands = []
                for p in sorted(local_runs_dir.glob("*/events.out.tfevents.*")):
                    evs = parse_tfevents_bytes(p.read_bytes())
                    rel_posix = p.relative_to(REPO_ROOT).as_posix()
                    cands.append((rel_posix, evs))
                    stream_inventory.append({
                        "Run": run_id,
                        "Stream Dir": p.parent.name,
                        "Events": len(evs),
                        "Max Step": max((e["step"] for e in evs), default=0),
                        "Eval Points": sum(1 for e in evs if e["tag"] == "eval/loss"),
                    })
                picked = _pick_best_completed_stream(cands)
                if picked and any(e["tag"] == "eval/loss" for e in picked[1]):
                    chosen_events = picked[1]
                    source_label = f"local:{picked[0]}"

        # 2. Git refs fallback
        if chosen_events is None and prefer_git:
            runs_prefix = f"models/adapters/{exp['adapter']}/runs"
            for ref in exp["git_refs"]:
                try:
                    ls_out = subprocess.check_output(
                        ["git", "ls-tree", "-r", "--name-only", ref, "--", runs_prefix],
                        cwd=REPO_ROOT,
                        stderr=subprocess.DEVNULL,
                    ).decode("utf-8", errors="replace").splitlines()
                except Exception:
                    continue
                ev_paths = [p for p in ls_out if "events.out.tfevents." in p]
                cands = []
                for rel_path in ev_paths:
                    try:
                        raw = subprocess.check_output(
                            ["git", "show", f"{ref}:{rel_path}"],
                            cwd=REPO_ROOT,
                            stderr=subprocess.DEVNULL,
                        )
                        cands.append((rel_path, parse_tfevents_bytes(raw)))
                    except Exception:
                        continue
                picked = _pick_best_completed_stream(cands)
                if picked and any(e["tag"] == "eval/loss" for e in picked[1]):
                    chosen_events = picked[1]
                    source_label = f"git:{ref}"
                    break

        # 3. Verified snapshot JSON fallback
        if chosen_events is None and run_id in snapshot:
            chosen_events = snapshot[run_id]["events"]
            source_label = f"snapshot:{snapshot[run_id].get('git_ref', 'saved')}"

        if chosen_events:
            sources[run_id] = source_label
            for ev in chosen_events:
                all_rows.append({
                    "run": run_id,
                    "step": ev["step"],
                    "tag": ev["tag"],
                    "value": ev["value"],
                    "wall_time": ev["wall_time"],
                })

    return pd.DataFrame(all_rows), sources, stream_inventory


def load_adapter_artifacts() -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for exp in EXPERIMENTS:
        adir = REPO_ROOT / "models" / "adapters" / exp["adapter"]
        sum_path = adir / "training_summary.json"
        prov_path = adir / "lexsi_provenance.json"
        final_dir = adir / "final_adapter"
        summary = json.loads(sum_path.read_text(encoding="utf-8")) if sum_path.is_file() else {}
        prov = json.loads(prov_path.read_text(encoding="utf-8")) if prov_path.is_file() else {}
        run_dirs = list((adir / "runs").glob("*/events.out.tfevents.*")) if (adir / "runs").is_dir() else []
        records.append({
            "Run": exp["run"],
            "Adapter": exp["adapter"],
            "Final Adapter": final_dir.is_dir(),
            "Summary JSON": sum_path.is_file(),
            "Run Attempts": len(run_dirs),
            "Saved At (UTC)": prov.get("created_at", "-"),
            "Summary LR": summary.get("learning_rate", np.nan),
        })
    return records


raw_df, data_sources, stream_inventory = load_all_experiments()
print(f"Repo root: {REPO_ROOT}")
print(f"Loaded {len(raw_df):,} scalar events across {raw_df['run'].nunique()} runs:")
for r, src in data_sources.items():
    n_ev = int((raw_df["run"] == r).sum())
    print(f"  - {r:<4} ({n_ev:>3} events) <- {src}")

In [ ]:
# Pivot raw scalar logs into tidy train & eval DataFrames
pivoted = (
    raw_df.sort_values(["run", "step", "wall_time"])
    .pivot_table(index=["run", "step"], columns="tag", values="value", aggfunc="last")
    .reset_index()
)
pivoted.columns.name = None

for meta_col in ["dataset_records", "unique_problems", "role"]:
    pivoted[meta_col] = pivoted["run"].map(lambda r: EXP_META[r][meta_col])

pivoted["effective_problem_passes"] = (pivoted["step"] * 16.0) / pivoted["unique_problems"]

train_df = pivoted[pivoted["train/loss"].notna()].copy()
eval_df = pivoted[pivoted["eval/loss"].notna()].copy()
if "train/epoch" not in eval_df.columns:
    eval_df["train/epoch"] = np.nan

eval_df["epoch"] = eval_df.groupby("run")["train/epoch"].transform(
    lambda s: (
        s.round().astype("int64")
        if s.notna().all()
        else pd.Series(np.arange(1, len(s) + 1, dtype="int64"), index=s.index)
    )
)

# Attach nearest preceding training step metrics for memorization gap
gap_rows = []
for run_id, eg in eval_df.groupby("run", sort=False):
    tg = train_df[train_df["run"] == run_id].sort_values("step")
    for _, erow in eg.iterrows():
        prev = tg[tg["step"] <= erow["step"]]
        last_t = prev.iloc[-1] if len(prev) > 0 else None
        train_loss_near = float(last_t["train/loss"]) if last_t is not None else np.nan
        train_acc_near = float(last_t["train/mean_token_accuracy"]) if last_t is not None else np.nan
        gap_rows.append({
            "run": run_id,
            "step": int(erow["step"]),
            "train_loss_near": train_loss_near,
            "train_acc_near": train_acc_near,
            "acc_gen_gap_pp": (train_acc_near - float(erow["eval/mean_token_accuracy"])) * 100.0,
            "loss_gen_gap": float(erow["eval/loss"]) - train_loss_near,
        })
gap_df = pd.DataFrame(gap_rows)
eval_df = eval_df.merge(gap_df, on=["run", "step"], how="left")

# Table 1: Validation & Generalization Summary
summary_records = []
compute_records = []
art_by_run = {a["Run"]: a for a in load_adapter_artifacts()}

for exp in EXPERIMENTS:
    r = exp["run"]
    esub = eval_df[eval_df["run"] == r].sort_values("epoch")
    tsub = train_df[train_df["run"] == r].sort_values("step")
    if len(esub) == 0:
        continue
    fin = (
        pivoted[(pivoted["run"] == r) & pivoted["train/train_loss"].notna()]
        if "train/train_loss" in pivoted.columns
        else pd.DataFrame()
    )
    best_loss_row = esub.loc[esub["eval/loss"].idxmin()]
    best_acc_row = esub.loc[esub["eval/mean_token_accuracy"].idxmax()]
    final_train_loss = float(fin["train/train_loss"].iloc[-1]) if len(fin) > 0 else float(tsub["train/loss"].iloc[-1])
    runtime_min = (
        float(fin["train/train_runtime"].iloc[-1]) / 60.0
        if len(fin) > 0 and "train/train_runtime" in fin.columns and pd.notna(fin["train/train_runtime"].iloc[-1])
        else np.nan
    )
    total_pflops = (
        float(fin["train/total_flos"].iloc[-1]) / 1e15
        if len(fin) > 0 and "train/total_flos" in fin.columns and pd.notna(fin["train/total_flos"].iloc[-1])
        else np.nan
    )
    samples_per_sec = (
        float(fin["train/train_samples_per_second"].iloc[-1])
        if len(fin) > 0 and "train/train_samples_per_second" in fin.columns and pd.notna(fin["train/train_samples_per_second"].iloc[-1])
        else np.nan
    )
    eval_rt_s = float(esub["eval/runtime"].mean()) if "eval/runtime" in esub.columns else np.nan
    ep1_tokens = float(esub["eval/num_tokens"].iloc[0]) if "eval/num_tokens" in esub.columns else np.nan
    art = art_by_run.get(r, {})

    summary_records.append({
        "Run": r,
        "Role": exp["role"],
        "Rows": exp["dataset_records"],
        "Uniq Probs": exp["unique_problems"],
        "Steps/Ep": int(esub["step"].iloc[0]),
        "Epochs": int(esub["epoch"].max()),
        "Best Ep (Loss)": int(best_loss_row["epoch"]),
        "Best eval/loss": round(float(best_loss_row["eval/loss"]), 4),
        "Last eval/loss": round(float(esub["eval/loss"].iloc[-1]), 4),
        "Best Ep (Acc)": int(best_acc_row["epoch"]),
        "Best eval/acc": round(float(best_acc_row["eval/mean_token_accuracy"]), 4),
        "Final train_loss": round(final_train_loss, 4),
        "Min Step loss": round(float(tsub["train/loss"].min()), 4),
        "Max Step acc": round(float(tsub["train/mean_token_accuracy"].max()), 4),
    })

    compute_records.append({
        "Run": r,
        "Final Adapter": art.get("Final Adapter", False),
        "Attempts": art.get("Run Attempts", 0),
        "Saved UTC": art.get("Saved At (UTC)", "-"),
        "Runtime (min)": round(runtime_min, 1) if pd.notna(runtime_min) else np.nan,
        "Eval RT (s/ep)": round(eval_rt_s, 1) if pd.notna(eval_rt_s) else np.nan,
        "Train Samp/s": round(samples_per_sec, 3) if pd.notna(samples_per_sec) else np.nan,
        "PFLOPs": round(total_pflops, 2) if pd.notna(total_pflops) else np.nan,
        "Tokens/Ep": int(round(ep1_tokens)) if pd.notna(ep1_tokens) else np.nan,
        "Avg GradNorm": round(float(tsub["train/grad_norm"].mean()), 3),
        "Max GradNorm": round(float(tsub["train/grad_norm"].max()), 3),
        "Init Train Ent": round(float(tsub["train/entropy"].iloc[0]), 3),
        "Min Train Ent": round(float(tsub["train/entropy"].min()), 3),
    })

summary_df = pd.DataFrame(summary_records)
compute_df = pd.DataFrame(compute_records)
print("=== Validation & Generalization Summary Across E1-E4B ===")
show_df(summary_df)
print(f"\n=== Merged Adapter Artifacts, Compute Cost & Training Stability (Total = {compute_df['Runtime (min)'].sum():.1f} min, {compute_df['PFLOPs'].sum():.2f} PFLOPs) ===")
show_df(compute_df)
if stream_inventory:
    print("\n=== Local TensorBoard Run Streams Under models/adapters/*/runs/ ===")
    show_df(pd.DataFrame(stream_inventory))

## 1. Validation Curves Across All Runs (`E1`–`E4B`)

Notice the structural split between the curves:
- **`E1` & `E2`** (raw textbook/PhysGym targets, not `aya-8b`-normalized): `eval/loss` reaches a clear minimum at **Epoch 2** (`1.4858` and `1.4666`) before rising. Note that `E1` completed 4 epochs cleanly (`Nov11_09-51-39`) after two earlier aborted attempts at steps `40` and `60`, then exited at post-training save without writing `final_adapter/`.
- **`E3`** (`aya-8b` canonicalized targets): `eval/loss` rises monotonically from **Epoch 1** (`1.5459 → 1.5504 → 1.5993 → 1.6571`), **yet `eval/mean_token_accuracy` improves at Epoch 2 (`0.6589 → 0.6622`)** while `eval/entropy` drops (`1.2716 → 1.2488`). That divergence is the signature of **output-format mismatch** against raw un-normalized PhysGym targets in `data/splits/validation.jsonl` (`0/38` contain `ANSWER:`).
- **`E4A`** (`aya-8b` + 2× completion repetition): starts at virtually the exact same Epoch 1 point as E3 (`1.5475`), then **degrades ~3× faster** in `eval/loss` (`1.6789 → 1.8764 → 1.9997`) AND **drops** in `eval/mean_token_accuracy` every epoch (`0.6611 → 0.6481 → 0.6345 → 0.6282`).
- **`E4B`** (`val_split_ratio=0.10` internal Verifier split): shows a textbook U-curve (`0.9145 → 0.8728 (Ep2) → 0.9121 → 1.0379`) on its ~`151–152` validation examples (`100.0s` eval runtime vs `~28–30s` for 38 Solver examples). *(Section 4 examines how random row-level splitting introduces `problem_id` overlap here.)*

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10), dpi=120)

# Panel A: Solver eval/loss
ax = axes[0, 0]
loss_ann_offsets = {
    "E1": (0.95, -0.031),
    "E2": (0.25, -0.092),
    "E3": (0.10, -0.131),
    "E4A": (0.08, 0.152),
}
for exp in SOLVERS:
    r_id = exp["run"]
    sub = eval_df[eval_df["run"] == r_id].sort_values("epoch")
    ax.plot(sub["epoch"], sub["eval/loss"], marker=exp["marker"], color=exp["color"], lw=2.2, ms=7, label=exp["label"])
    best = sub.loc[sub["eval/loss"].idxmin()]
    ax.scatter([best["epoch"]], [best["eval/loss"]], s=190, marker="*", color=exp["color"], edgecolor="black", zorder=6)
    dx, dy = loss_ann_offsets[r_id]
    ax.annotate(
        f"{r_id}: Ep{int(best['epoch'])} ({best['eval/loss']:.3f})",
        xy=(best["epoch"], best["eval/loss"]),
        xytext=(best["epoch"] + dx, best["eval/loss"] + dy),
        arrowprops=dict(arrowstyle="->", color=exp["color"], lw=1.0),
        fontsize=8.6, color=exp["color"], fontweight="bold",
    )
ax.set_title("A. Solver Validation Loss (eval/loss) by Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("eval/loss (raw PhysGym validation.jsonl)")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_xlim(0.75, 5.25)
ax.set_ylim(1.34, 2.08)
ax.legend(loc="upper left", fontsize=9)

# Panel B: Solver eval/mean_token_accuracy
ax = axes[0, 1]
for exp in SOLVERS:
    sub = eval_df[eval_df["run"] == exp["run"]].sort_values("epoch")
    ax.plot(sub["epoch"], sub["eval/mean_token_accuracy"], marker=exp["marker"], color=exp["color"], lw=2.2, ms=7, label=exp["run"])
e3_ep2 = eval_df[(eval_df["run"] == "E3") & (eval_df["epoch"] == 2)]
e4a_ep2 = eval_df[(eval_df["run"] == "E4A") & (eval_df["epoch"] == 2)]
if len(e3_ep2) > 0:
    row = e3_ep2.iloc[0]
    ax.annotate(
        f"E3 Ep2 rises ({row['eval/mean_token_accuracy']:.4f})",
        xy=(2, row["eval/mean_token_accuracy"]),
        xytext=(2.45, 0.6708),
        arrowprops=dict(arrowstyle="->", color="#10b981", lw=1.2),
        fontsize=9, fontweight="bold", color="#047857",
    )
if len(e4a_ep2) > 0:
    row = e4a_ep2.iloc[0]
    ax.annotate(
        f"E4A Ep2 collapses ({row['eval/mean_token_accuracy']:.4f})",
        xy=(2, row["eval/mean_token_accuracy"]),
        xytext=(2.45, 0.6485),
        arrowprops=dict(arrowstyle="->", color="#ef4444", lw=1.2),
        fontsize=9, fontweight="bold", color="#b91c1c",
    )
ax.set_title("B. Solver Validation Token Accuracy (eval/mean_token_accuracy)")
ax.set_xlabel("Epoch")
ax.set_ylabel("eval/mean_token_accuracy")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_ylim(0.620, 0.678)
ax.legend(loc="lower left", fontsize=9)

# Panel C: Validation Prediction Entropy
ax = axes[1, 0]
for exp in EXPERIMENTS:
    sub = eval_df[eval_df["run"] == exp["run"]].sort_values("epoch")
    ls = "--" if exp["role"] == "verifier" else "-"
    ax.plot(sub["epoch"], sub["eval/entropy"], marker=exp["marker"], ls=ls, color=exp["color"], lw=2.1, ms=6.5, label=exp["run"])
ax.set_title("C. Validation Prediction Entropy (eval/entropy) by Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("eval/entropy")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend(loc="upper right", fontsize=9)

# Panel D: E4B Verifier In-Distribution Validation (dual axis)
ax = axes[1, 1]
e4b_sub = eval_df[eval_df["run"] == "E4B"].sort_values("epoch")
l1 = ax.plot(e4b_sub["epoch"], e4b_sub["eval/loss"], marker="P", color="#8b5cf6", lw=2.3, ms=8, label="E4B eval/loss (left)")
best_b = e4b_sub.loc[e4b_sub["eval/loss"].idxmin()]
ax.scatter([best_b["epoch"]], [best_b["eval/loss"]], s=180, marker="*", color="#8b5cf6", edgecolor="black", zorder=6)
ax.annotate(
    f"Clean Ep{int(best_b['epoch'])} U-turn ({best_b['eval/loss']:.4f})",
    xy=(best_b["epoch"], best_b["eval/loss"]),
    xytext=(best_b["epoch"] + 0.35, best_b["eval/loss"] + 0.025),
    arrowprops=dict(arrowstyle="->", color="#6d28d9"),
    fontsize=9, fontweight="bold", color="#6d28d9",
)
ax.set_ylabel("E4B eval/loss (10% internal split)", color="#6d28d9")
ax.set_xlabel("Epoch")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_ylim(0.84, 1.07)

ax2 = ax.twinx()
ax2.grid(False)
l2 = ax2.plot(e4b_sub["epoch"], e4b_sub["eval/mean_token_accuracy"], marker="o", ls="--", color="#ec4899", lw=2.0, ms=6.5, label="E4B eval/token_acc (right)")
ax2.set_ylabel("E4B eval/mean_token_accuracy", color="#be185d")
ax2.set_ylim(0.73, 0.765)
lines = l1 + l2
ax.legend(lines, [l.get_label() for l in lines], loc="upper left", fontsize=9)
ax.set_title("D. Verifier E4B Control: In-Distribution 10% Validation Curve")

plt.tight_layout()
plt.show()

## 2. Step-Matched & Target-Exposure Alignment (`E4A` vs `E3`) and LR Dynamics

Why did `E4A` diverge **~3× faster per epoch** than `E3`?
- **`E3`** has `681` rows (`43` steps/epoch).
- **`E4A`** has `1,444` rows (`91` steps/epoch), where **all `681` canonical `target_output` derivations appear twice per epoch** (`turn0_sol1` + either `turn1_1hint` or `turn2_2hints`), plus `82` minority `turn0_sol2` variants.
- Because `completion_only_loss=True` masks prompts/hints with `-100`, **1 epoch of `E4A` (`91` steps, `874,940` tokens) executes ~2.1× as many gradient updates over the exact same 681 target solutions as 1 epoch of `E3` (`43` steps, `369,243` tokens)**:
  - **`E4A` Epoch 1 (step 91)** (`eval_loss = 1.5475`, `acc = 0.6611`, `entropy = 1.2530`) aligns almost directly with **`E3` Epoch 2 (step 86)** (`1.5504`, `0.6622`, `1.2488`).
  - **`E4A` Epoch 2 (step 182)** (`eval_loss = 1.6789`, `acc = 0.6481`, `entropy = 1.0765`) aligns right next to **`E3` Epoch 4 (step 172)** (`1.6571`, `0.6527`, `1.0470`).
  - Meanwhile, `warmup_ratio = 0.10` over `10` epochs (`910` steps) places peak LR (`0.0002`) right at **Step 91** (start of E4A Epoch 2), driving E4A down to min step `train/loss = 0.112` (`96.76%` peak train token accuracy; `96.35%` at step 360 vs `62.82%` val at Epoch 4). Also, `cosine_with_restarts` ends Cycle 1 at Epoch 4 (`40%` of total steps) and restarts back to peak LR at Epoch 4.1 (`E2` step 190 → `1.99e-4`, `E4B` step 350 → `1.50e-4`).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10), dpi=120)

# Panel A: train/loss vs Optimizer Step
ax = axes[0, 0]
for exp in EXPERIMENTS:
    sub = train_df[train_df["run"] == exp["run"]].sort_values("step")
    ls = "--" if exp["role"] == "verifier" else "-"
    ax.plot(sub["step"], sub["train/loss"], ls=ls, color=exp["color"], lw=2.0, label=exp["run"])
ax.axvline(91, color="#ef4444", ls=":", lw=1.4, alpha=0.8)
ax.text(95, 2.30, "E4A Ep1 ends\n(step 91)", color="#b91c1c", fontsize=8.8)
ax.set_title("A. Training Loss (train/loss) vs Optimizer Step")
ax.set_xlabel("Optimizer Step (logged every 10 steps)")
ax.set_ylabel("train/loss")
ax.legend(loc="upper right", fontsize=9)

# Panel B: Learning Rate Schedule
ax = axes[0, 1]
for exp in EXPERIMENTS:
    sub = train_df[train_df["run"] == exp["run"]].sort_values("step")
    ls = "--" if exp["role"] == "verifier" else "-"
    ax.plot(sub["step"], sub["train/learning_rate"] * 1e4, ls=ls, color=exp["color"], lw=2.0, label=exp["run"])
e2_restart = train_df[(train_df["run"] == "E2") & (train_df["step"] == 190)]
if len(e2_restart) > 0:
    y_val = float(e2_restart["train/learning_rate"].iloc[0]) * 1e4
    ax.annotate(
        "Cycle 1 hits 0 at Ep4 (40%)\nthen restarts to peak LR (E2, E4B)",
        xy=(190, y_val), xytext=(245, 1.75),
        arrowprops=dict(arrowstyle="->", color="#0284c7"),
        fontsize=8.8, color="#0369a1", fontweight="bold",
    )
ax.annotate(
    "E4A warmup peaks\nright at Ep1 -> Ep2 (step 91)",
    xy=(95, 2.0), xytext=(110, 2.13),
    arrowprops=dict(arrowstyle="->", color="#dc2626"),
    fontsize=8.8, color="#b91c1c", fontweight="bold",
)
ax.set_title("B. Learning Rate Schedule (×10⁻⁴) vs Optimizer Step")
ax.set_xlabel("Optimizer Step")
ax.set_ylabel("Learning Rate (×10⁻⁴)")
ax.set_ylim(-0.08, 2.38)
ax.legend(loc="lower right", fontsize=9)

# Panel C: eval/loss vs Effective Passes Over Unique Problems (Solvers)
ax = axes[1, 0]
for exp in SOLVERS:
    sub = eval_df[eval_df["run"] == exp["run"]].sort_values("step")
    ax.plot(sub["effective_problem_passes"], sub["eval/loss"], marker=exp["marker"], color=exp["color"], lw=2.3, ms=7.5, label=exp["run"])
    for _, r in sub.iterrows():
        ep = int(r["epoch"])
        if exp["run"] == "E3" and ep in (1, 2, 4):
            ax.text(r["effective_problem_passes"] + 0.10, r["eval/loss"] - 0.036, f"E3 E{ep}", fontsize=8.4, color=exp["color"], fontweight="bold")
        elif exp["run"] == "E4A" and ep in (1, 2, 3, 4):
            ax.text(r["effective_problem_passes"] - 0.35, r["eval/loss"] + 0.022, f"E4A E{ep}", fontsize=8.4, color=exp["color"], fontweight="bold")
ax.set_title("C. E3 vs E4A Collapse Alignment by Passes Over Unique Problems")
ax.set_xlabel("Effective Problem Passes (step × 16 / unique_problems)")
ax.set_ylabel("eval/loss")
ax.set_ylim(1.42, 2.07)
ax.legend(loc="upper left", fontsize=9)

# Panel D: Train - Val Token Accuracy Generalization Gap
ax = axes[1, 1]
for exp in EXPERIMENTS:
    sub = eval_df[eval_df["run"] == exp["run"]].sort_values("epoch")
    ls = "--" if exp["role"] == "verifier" else "-"
    ax.plot(sub["epoch"], sub["acc_gen_gap_pp"], marker=exp["marker"], ls=ls, color=exp["color"], lw=2.2, ms=7, label=exp["run"])
e4a_sub = eval_df[eval_df["run"] == "E4A"].sort_values("epoch")
if len(e4a_sub) > 0:
    e4a_last = e4a_sub.iloc[-1]
    ax.annotate(
        f"E4A Ep{int(e4a_last['epoch'])} gap: +{e4a_last['acc_gen_gap_pp']:.1f} pp\n({e4a_last['train_acc_near']*100:.1f}% train vs {e4a_last['eval/mean_token_accuracy']*100:.1f}% val)",
        xy=(e4a_last["epoch"], e4a_last["acc_gen_gap_pp"]),
        xytext=(2.05, e4a_last["acc_gen_gap_pp"] - 5.5),
        arrowprops=dict(arrowstyle="->", color="#dc2626"),
        fontsize=9, fontweight="bold", color="#b91c1c",
    )
ax.axhline(0, color="#94a3b8", lw=1, ls="-")
ax.set_title("D. Overfitting Gap (Train Token Acc − Val Token Acc, pp)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Generalization Gap (percentage points)")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

## 3. Training Stability, Entropy Collapse & Compute Footprint (`grad_norm`, `entropy`, `FLOPs`)

Now that all 5 adapters are merged locally (`models/adapters/*`), we can inspect optimization stability and compute cost alongside `eval/loss`:
- **Target heterogeneity vs style canonicalization (`train/grad_norm` & `train/entropy`):**
  - Raw textbook targets (`E1`, `E2`) begin at **`train/entropy ≈ 2.02–2.05`** and exhibit **~3–4× larger gradient norms** (`avg = 1.11` & `1.05`, peaking at `1.92` & `2.19`) because step styles and LaTeX vary wildly across problems.
  - Canonical `aya-8b` targets (`E3`, `E4A`) start at **`train/entropy ≈ 0.94–0.95`** (already near-template predictability at step `10`), with tiny gradients (`avg = 0.29` & `0.40`). In `E4A`, `train/entropy` plunges to **`0.166`**, signaling aggressive surface memorization.
- **`cosine_with_restarts` LR restart shocks at Epoch 4.1:** Both runs that passed Epoch 4 (`E2` step `180 → 190` and `E4B` step `340 → 350`) experience an immediate gradient spike (`1.33 → 2.19` in `E2`, `1.01 → 1.525` in `E4B`) right when LR snaps from `~1e-7–1e-8` back up to peak LR (`2.0e-4` / `1.5e-4`).
- **Total compute budget:** Across the 5 completed training runs, total GPU training time was **`486.9` min (`8.12` hrs)** and **`209.0` PFLOPs** (`E2` & `E3` ran concurrently across dual GPUs `11:28–12:20Z`; `E4A` & `E4B` ran concurrently `12:20–16:03Z`).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10), dpi=120)

# Panel A: train/grad_norm vs Optimizer Step
ax = axes[0, 0]
for exp in EXPERIMENTS:
    sub = train_df[train_df["run"] == exp["run"]].sort_values("step")
    ls = "--" if exp["role"] == "verifier" else "-"
    ax.plot(sub["step"], sub["train/grad_norm"], ls=ls, color=exp["color"], lw=1.9, alpha=0.9, label=exp["run"])
e2_pk = train_df[(train_df["run"] == "E2") & (train_df["step"] == 230)]
e4b_pk = train_df[(train_df["run"] == "E4B") & (train_df["step"] == 360)]
if len(e2_pk) > 0:
    y2 = float(e2_pk["train/grad_norm"].iloc[0])
    ax.annotate(f"E2 restart spike\n({y2:.2f})", xy=(230, y2), xytext=(248, y2 - 0.15), arrowprops=dict(arrowstyle="->", color="#0ea5e9"), fontsize=8.6, fontweight="bold", color="#0369a1")
if len(e4b_pk) > 0:
    y4 = float(e4b_pk["train/grad_norm"].iloc[0])
    ax.annotate(f"E4B restart spike\n({y4:.2f})", xy=(360, y4), xytext=(315, y4 + 0.25), arrowprops=dict(arrowstyle="->", color="#8b5cf6"), fontsize=8.6, fontweight="bold", color="#6d28d9")
ax.set_title("A. Gradient L2 Norm (train/grad_norm) vs Optimizer Step")
ax.set_xlabel("Optimizer Step")
ax.set_ylabel("train/grad_norm")
ax.set_ylim(0.1, 2.40)
ax.legend(loc="upper left", fontsize=9)

# Panel B: train/entropy vs Optimizer Step
ax = axes[0, 1]
for exp in EXPERIMENTS:
    sub = train_df[train_df["run"] == exp["run"]].sort_values("step")
    ls = "--" if exp["role"] == "verifier" else "-"
    ax.plot(sub["step"], sub["train/entropy"], ls=ls, color=exp["color"], lw=2.1, label=exp["run"])
e4a_min = train_df[train_df["run"] == "E4A"].sort_values("train/entropy").iloc[0]
ax.annotate(
    f"E4A collapses to {e4a_min['train/entropy']:.3f}",
    xy=(e4a_min["step"], e4a_min["train/entropy"]),
    xytext=(215, 0.36),
    arrowprops=dict(arrowstyle="->", color="#ef4444"),
    fontsize=8.8, fontweight="bold", color="#b91c1c",
)
ax.set_title("B. Training Token Entropy (train/entropy) vs Optimizer Step")
ax.set_xlabel("Optimizer Step")
ax.set_ylabel("train/entropy")
ax.legend(loc="upper right", fontsize=9)

# Panel C: Train Token Accuracy vs Optimizer Step
ax = axes[1, 0]
for exp in EXPERIMENTS:
    sub = train_df[train_df["run"] == exp["run"]].sort_values("step")
    ls = "--" if exp["role"] == "verifier" else "-"
    ax.plot(sub["step"], sub["train/mean_token_accuracy"], ls=ls, color=exp["color"], lw=2.1, label=exp["run"])
ax.axhline(0.90, color="#94a3b8", ls=":", lw=1.2)
ax.text(12, 0.906, "90% train token acc threshold", color="#64748b", fontsize=8.5)
ax.set_title("C. Training Token Accuracy Saturation vs Optimizer Step")
ax.set_xlabel("Optimizer Step")
ax.set_ylabel("train/mean_token_accuracy")
ax.set_ylim(0.42, 0.99)
ax.legend(loc="lower right", fontsize=9)

# Panel D: Compute Footprint (Runtime vs PFLOPs, sized/labeled by throughput)
ax = axes[1, 1]
compute_offsets = {
    "E1": (14, 0.5),
    "E2": (10, -6.5),
    "E3": (-12, 9.0),
    "E4A": (10, -3.0),
    "E4B": (-74, -9.0),
}
for _, row in compute_df.iterrows():
    meta = EXP_META[row["Run"]]
    ax.scatter(row["Runtime (min)"], row["PFLOPs"], s=row["Tokens/Ep"] / 3500.0 + 90.0, color=meta["color"], marker=meta["marker"], edgecolor="black", alpha=0.88, zorder=5)
    dx, dy = compute_offsets.get(row["Run"], (6, -4))
    ax.annotate(
        f"{row['Run']}: {row['Runtime (min)']:.0f}m, {row['PFLOPs']:.1f} PF\n({row['Train Samp/s']:.2f} samp/s)",
        xy=(row["Runtime (min)"], row["PFLOPs"]),
        xytext=(row["Runtime (min)"] + dx, row["PFLOPs"] + dy),
        fontsize=8.6, fontweight="bold", color=meta["color"],
    )
ax.set_title("D. Compute Cost Frontier (Bubble Area ∝ Tokens/Epoch)")
ax.set_xlabel("Train Runtime (minutes)")
ax.set_ylabel("Total Training Compute (PFLOPs, 10¹⁵)")
ax.set_xlim(25, 250)
ax.set_ylim(5, 102)

plt.tight_layout()
plt.show()

## 4. Dataset Composition, Domain Shift & Verifier Split Audit (`E1`–`E4B` vs `Val/Test`)

Inspecting all Stage 2.5 JSONL splits (`data/sft/*`) and held-out PhysGym splits (`data/splits/{validation,test}.jsonl`) highlights four structural drivers behind the curves above:
- **Domain & length shift:** Textbook training splits (`E1`–`E4B`) contain `51.8%–58.4%` `MECHANICS` and only `3.1%–4.9%` `THERMODYNAMICS`, whereas held-out PhysGym `Val`/`Test` are `39.5%` `MECHANICS`, `31.6%` `ELECTROMAGNETISM`, and `10.5%` `THERMODYNAMICS`.
- **`E4A` completion duplication & format mix:** `681` byte-identical `target_output` strings repeat across `turn0_sol1` and `turn1_1hint`/`turn2_2hints` (`47.2%` hint rows), alongside `80/82` `turn0_sol2` variants starting with `Step 1:` and `150` multi-`ANSWER:` rows.
- **`E4B` class imbalance & split leakage risk:** `verifier_e4_curated` has `1,331` `[[CHALLENGE]]` (`88.1%`) vs `180` `[[APPROVE]]` (`11.9%`), and **`96.3%` of problems appear `2–4×`** (`681` unique `problem_id`s across `1,511` rows), creating ~`91%` problem-level overlap under row-random `val_split_ratio=0.10`.

In [ ]:
# Audit ALL SFT datasets (E1-E4B) vs held-out PhysGym validation/test splits directly from data/
def load_jsonl_rows(rel_paths: list[str]) -> list[dict[str, Any]]:
    for rel_path in rel_paths:
        p = REPO_ROOT / rel_path
        if p.is_file():
            with p.open("r", encoding="utf-8") as f:
                return [json.loads(line) for line in f if line.strip()]
    return []

dataset_specs = [
    ("E1 Train", ["data/sft/solver_e1_manual_det/train.jsonl"], "prompt", "target_output"),
    ("E2 Train", ["data/sft/solver_e2_curatorkit_nollm/train.jsonl"], "prompt", "target_output"),
    ("E3 Train", ["data/sft/solver_e3_canonical_llm/train.jsonl"], "prompt", "target_output"),
    ("E4A Train", ["data/sft/solver_e4_adversarial/train.jsonl"], "prompt", "target_output"),
    ("E4B Train", ["data/sft/verifier_e4_curated/train.jsonl"], "prompt", "target_output"),
    ("Val Split (38)", ["data/sft/solver_e4_adversarial/validation.jsonl", "data/splits/validation.jsonl"], "prompt", "target_output"),
    ("Test Split (38)", ["data/sft/solver_e4_adversarial/test.jsonl", "data/splits/test.jsonl"], "prompt", "target_output"),
]

DOMAINS = ["MECHANICS", "ELECTROMAGNETISM", "MODERN_PHYSICS", "THERMODYNAMICS", "OPTICS", "RELATIVITY"]
audit_rows = []
domain_pct_rows = []
e4a_rows_raw: list[dict[str, Any]] = []
e4b_rows_raw: list[dict[str, Any]] = []
target_lengths_by_group: dict[str, list[int]] = {}
prompt_lengths_by_group: dict[str, list[int]] = {}

for name, rel_paths, prompt_key, target_key in dataset_specs:
    rows = load_jsonl_rows(rel_paths)
    if not rows:
        continue
    if name == "E4A Train":
        e4a_rows_raw = rows
    elif name == "E4B Train":
        e4b_rows_raw = rows

    p_lens = [len(str(r.get(prompt_key) or r.get("problem_statement") or "")) for r in rows]
    t_lens = [len(str(r.get(target_key) or r.get("reference_solution") or "")) for r in rows]
    t_strs = [str(r.get(target_key) or r.get("reference_solution") or "") for r in rows]
    phys = sum(1 for r in rows if str(r.get("problem_id", "")).startswith("physgym"))
    uniq_p = len({str(r.get("problem_id", idx)) for idx, r in enumerate(rows)})
    has_ans = sum(1 for t in t_strs if "ANSWER:" in t)
    multi_ans = sum(1 for t in t_strs if len(re.findall(r"ANSWER:", t)) > 1)
    step1 = sum(1 for t in t_strs if t.lstrip().startswith("Step 1:"))
    prompt_lengths_by_group[name] = p_lens
    target_lengths_by_group[name] = t_lens

    dom_counts = Counter(str(r.get("domain", "UNKNOWN")) for r in rows)
    dom_row: dict[str, Any] = {"Split": name}
    for d in DOMAINS:
        dom_row[d] = round(100.0 * dom_counts.get(d, 0) / len(rows), 1)
    domain_pct_rows.append(dom_row)

    audit_rows.append({
        "Split": name,
        "Count": len(rows),
        "Uniq Probs": uniq_p,
        "PhysGym (%)": round(100.0 * phys / len(rows), 1),
        "Prompt Avg Chars": int(round(np.mean(p_lens))),
        "Target Avg Chars": int(round(np.mean(t_lens))),
        "Target Max Chars": int(np.max(t_lens)),
        "Has ANSWER: (%)": round(100.0 * has_ans / len(rows), 1),
        "Multi-ANSWER": multi_ans,
        "Starts 'Step 1:'": step1,
    })

audit_df = pd.DataFrame(audit_rows)
domain_pct_df = pd.DataFrame(domain_pct_rows)
print("=== Dataset Composition & Target Format Audit (E1-E4B & PhysGym Val/Test) ===")
show_df(audit_df)
print("\n=== Domain Share (%) Across Splits ===")
show_df(domain_pct_df)

# Deep-dive into E4A internal composition & duplication
e4a_slice_df = pd.DataFrame()
if e4a_rows_raw:
    sol1_targets = {
        r["problem_id"]: r["target_output"]
        for r in e4a_rows_raw
        if "turn0_sol1" in str(r.get("trajectory_id", ""))
    }
    slice_map: dict[str, list[dict[str, Any]]] = {
        "turn0_sol1": [],
        "turn0_sol2": [],
        "turn1_1hint": [],
        "turn2_2hints": [],
    }
    for r in e4a_rows_raw:
        tid = str(r.get("trajectory_id", ""))
        if "turn0_sol2" in tid:
            slice_map["turn0_sol2"].append(r)
        elif int(r.get("turn", 0)) == 0:
            slice_map["turn0_sol1"].append(r)
        elif int(r.get("turn", 0)) == 1:
            slice_map["turn1_1hint"].append(r)
        else:
            slice_map["turn2_2hints"].append(r)

    slice_stats = []
    for sname, srows in slice_map.items():
        dup_vs_sol1 = sum(1 for r in srows if r["target_output"] == sol1_targets.get(r["problem_id"]))
        step1_cnt = sum(1 for r in srows if str(r["target_output"]).lstrip().startswith("Step 1:"))
        tot_lens = [len(r["prompt"]) + len(r["target_output"]) for r in srows]
        target_lengths_by_group[f"E4A {sname}"] = [len(r["target_output"]) for r in srows]
        slice_stats.append({
            "E4A Slice": sname,
            "Rows": len(srows),
            "Share (%)": round(100.0 * len(srows) / len(e4a_rows_raw), 1),
            "Identical Target to Sol1": dup_vs_sol1 if sname != "turn0_sol1" else len(srows),
            "Starts 'Step 1:'": step1_cnt,
            "Multi-ANSWER": sum(1 for r in srows if len(re.findall(r"ANSWER:", r["target_output"])) > 1),
            "Avg Total Chars": int(round(np.mean(tot_lens))) if tot_lens else 0,
            "Max Total Chars": int(np.max(tot_lens)) if tot_lens else 0,
        })
    e4a_slice_df = pd.DataFrame(slice_stats)
    print("\n=== E4A Internal Slice Breakdown ===")
    show_df(e4a_slice_df)
    prompt_dups = sum(1 for _, c in Counter(r["prompt"] for r in e4a_rows_raw).items() if c > 1)
    target_dups = sum(1 for _, c in Counter(r["target_output"] for r in e4a_rows_raw).items() if c > 1)
    print(f"Identical prompts appearing >1x in E4A: {prompt_dups} | Identical targets appearing >1x: {target_dups}")

# Deep-dive into E4B Verifier composition & random row-split leakage risk
e4b_err_df = pd.DataFrame()
e4b_leak_pct = np.nan
if e4b_rows_raw:
    verdict_counts = Counter(str(r.get("verdict", "")) for r in e4b_rows_raw)
    err_counts = Counter((str(r.get("error_type") or "approved_no_error")) for r in e4b_rows_raw)
    e4b_err_df = pd.DataFrame([
        {
            "Category": k,
            "Count": v,
            "Share (%)": round(100.0 * v / len(e4b_rows_raw), 1),
        }
        for k, v in err_counts.most_common()
    ])
    pid_mult = Counter(Counter(str(r.get("problem_id", "")) for r in e4b_rows_raw).values())
    pids_arr = np.array([str(r.get("problem_id", "")) for r in e4b_rows_raw])
    n_val = int(round(0.10 * len(pids_arr)))  # ~151 rows (matches 85*16=1360 train -> 151 val)
    rng = np.random.default_rng(42)
    overlaps = []
    for _ in range(500):
        perm = rng.permutation(len(pids_arr))
        val_pids = pids_arr[perm[:n_val]]
        tr_pid_set = set(pids_arr[perm[n_val:]])
        overlaps.append(100.0 * sum(1 for pid in val_pids if pid in tr_pid_set) / len(val_pids))
    e4b_leak_pct = float(np.mean(overlaps))
    print("\n=== E4B Verifier Composition & Problem-Overlap Audit ===")
    print(f"Verdicts: {dict(verdict_counts)} | Problem multiplicity (rows/problem -> count): {dict(sorted(pid_mult.items()))}")
    print(f"Expected val rows whose problem_id ALSO appears in train under 10% row-random split (500 MC trials): {e4b_leak_pct:.1f}%")
    show_df(e4b_err_df)

In [ ]:
if len(audit_df) > 0 and len(e4a_slice_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16.5, 11.2), dpi=120)

    # Panel A: Domain Shift Across Splits (Stacked Horizontal Bars)
    ax = axes[0, 0]
    dom_palette = {
        "MECHANICS": "#0ea5e9",
        "ELECTROMAGNETISM": "#f97316",
        "MODERN_PHYSICS": "#10b981",
        "THERMODYNAMICS": "#ef4444",
        "OPTICS": "#8b5cf6",
        "RELATIVITY": "#eab308",
    }
    rev_dom = domain_pct_df.iloc[::-1].reset_index(drop=True)
    left = np.zeros(len(rev_dom))
    for d in DOMAINS:
        vals = rev_dom[d].to_numpy()
        ax.barh(rev_dom["Split"], vals, left=left, color=dom_palette[d], edgecolor="white", linewidth=0.5, label=d)
        for idx, v in enumerate(vals):
            if v >= 7.5:
                ax.text(left[idx] + v / 2.0, idx, f"{v:.0f}%", ha="center", va="center", color="white", fontsize=8.2, fontweight="bold")
        left += vals
    ax.set_xlim(0, 100)
    ax.set_xlabel("Domain Share (%)")
    ax.set_title("A. Domain Distribution Shift (Train vs PhysGym Val/Test)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=8.5)

    # Panel B: Average Prompt + Target Length Budget Explaining Compute Differences
    ax = axes[0, 1]
    x = np.arange(len(audit_df))
    p_avgs = audit_df["Prompt Avg Chars"].to_numpy()
    t_avgs = audit_df["Target Avg Chars"].to_numpy()
    ax.bar(x, p_avgs, color="#64748b", label="Avg Prompt Chars (masked -100)")
    ax.bar(x, t_avgs, bottom=p_avgs, color="#0ea5e9", label="Avg Target Completion Chars")
    for xi, (pv, tv) in enumerate(zip(p_avgs, t_avgs)):
        ax.text(xi, pv + tv + 55, f"{int(pv + tv)}c", ha="center", fontsize=8.4, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(audit_df["Split"], rotation=25, ha="right", fontsize=9)
    ax.set_ylabel("Average Character Length per Example")
    ax.set_ylim(0, max(p_avgs + t_avgs) + 420)
    ax.set_title("B. Sequence Budget: Why E4B Runs 2.6× Slower per Sample")
    ax.legend(loc="upper left", fontsize=8.8)

    # Panel C: E4A Slice Composition & Target Duplication
    ax = axes[1, 0]
    slice_notes = {
        "turn0_sol1": "Canonical baseline (681)",
        "turn0_sol2": "80/82 'Step 1:' format + dup prompt",
        "turn1_1hint": "100% byte-identical target to Sol1",
        "turn2_2hints": "100% byte-identical target to Sol1",
    }
    slice_colors = {
        "turn0_sol1": "#10b981",
        "turn0_sol2": "#f59e0b",
        "turn1_1hint": "#ef4444",
        "turn2_2hints": "#b91c1c",
    }
    rev_df = e4a_slice_df.iloc[::-1].reset_index(drop=True)
    bars = ax.barh(
        rev_df["E4A Slice"],
        rev_df["Rows"],
        color=[slice_colors[s] for s in rev_df["E4A Slice"]],
        edgecolor="#1e293b",
        linewidth=0.6,
    )
    for bar, sname in zip(bars, rev_df["E4A Slice"]):
        val = bar.get_width()
        ax.text(val + 16, bar.get_y() + bar.get_height() / 2, f"{int(val)}  ({slice_notes[sname]})", va="center", fontsize=8.5)
    ax.set_xlim(0, 1480)
    ax.set_xlabel("Training Examples in E4A (Total = 1,444)")
    ax.set_title("C. E4A Composition: 47.2% Hints + 2× Identical Targets")

    # Panel D: E4B Verifier Verdict Imbalance & Error Types
    ax = axes[1, 1]
    if len(e4b_err_df) > 0:
        rev_err = e4b_err_df.iloc[::-1].reset_index(drop=True)
        err_colors = ["#10b981" if c == "approved_no_error" else ("#ef4444" if c == "real_model_a_failure" else "#8b5cf6") for c in rev_err["Category"]]
        ebars = ax.barh(rev_err["Category"], rev_err["Count"], color=err_colors, edgecolor="#1e293b", linewidth=0.6)
        for bar, share in zip(ebars, rev_err["Share (%)"]):
            val = bar.get_width()
            ax.text(val + 12, bar.get_y() + bar.get_height() / 2, f"{int(val)} ({share:.1f}%)", va="center", fontsize=8.6)
        ax.set_xlim(0, 860)
        ax.set_xlabel("Examples in E4B Verifier Train Set (Total = 1,511)")
        leak_note = f"~{e4b_leak_pct:.0f}% row-split problem overlap" if pd.notna(e4b_leak_pct) else "10% random split"
        ax.set_title(f"D. E4B Composition: 88.1% CHALLENGE vs 11.9% APPROVE ({leak_note})")

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.38)
    plt.show()
else:
    print("Local data/sft or data/splits files not present in this checkout; skipping Dataset Audit plots.")

## 5. Synthesis & Concrete Next Steps

| Lever | Effect Verified from Merged Results & Data | Action |
| :--- | :--- | :--- |
| **1. Missing `E1` Final Adapter & Artifact Hygiene** | `E1` crashed at `trainer.save_model(path=...)` (`13fa48a`) and lacks `final_adapter/` and `training_summary.json`; only `E2`, `E3`, `E4A`, `E4B` saved complete artifacts (weights live on HF Hub). | Re-run/recover `E1` only if needed as a raw-deterministic baseline after benchmarking `E2`, `E3`, `E4A` from HF Hub. |
| **2. Early-Stopping Metric (`eval_loss` → Task / In-Dist)** | `eval/loss` on raw PhysGym `validation.jsonl` (`0/38` `ANSWER:`) penalizes `aya-8b` formatting (`E3` Ep2 accuracy rose while `eval_loss` rose). | Save epochs `1..3` and select solver checkpoints offline by `final_answer_accuracy` on `validation.jsonl`, **or** use a `problem_id`-grouped `10%` canonical holdout for early stopping. |
| **3. Benchmark `E3` vs `E4A + E4B` Together First** | `47.2%` of `E4A` trains hint-conditioned revision (`CRITICAL PHYSICAL CONSTRAINTS TO FOLLOW:`). Turn-0 alone ignores `E4A`'s multi-turn design. | Evaluate `E2`, `E3`, and `E4A` on Stage 5 (`final_answer_accuracy`) at **Turn 0 AND multi-turn with `E4B`** (`verifier_e4_curated_lora`) before retraining. |
| **4. Fix Schedule & LR Restart Shocks** | Warmup (`0.10`) peaks right at `E4A` `Ep1 → Ep2` (`step 91`), driving `train_acc` to `96.8%` peak (`train/entropy → 0.166`); `cosine_with_restarts` causes `grad_norm` spikes at Ep4.1 (`E2` `2.19`, `E4B` `1.53`). | Use standard **`cosine`** decay (no Ep4 cycle reset), **`num_train_epochs=3–4`**, **`warmup_ratio=0.05`**, lower LR (`1.0e-4` or warm-start `E4A` from `E3` at `5e-5`), and `lora_dropout=0.10`. |
| **5. `E4A` Cleanup & `E4B` Grouped Split / Balance** | `E4A` repeats `681` byte-identical targets + has `80/82` `turn0_sol2` starting with `Step 1:` + `150` multi-`ANSWER:` rows. `E4B` is `88.1%` `[[CHALLENGE]]` / `11.9%` `[[APPROVE]]` and `96.3%` of problems appear `2–4×` (~`91%` problem overlap under row-random `val_split_ratio=0.10`). | Unify `E4A` `turn0_sol2` formatting, strip extra `ANSWER:` lines, subsample/diversify hint targets, switch `E4B` to a **`problem_id`-grouped validation split**, and balance `[[APPROVE]]` vs `[[CHALLENGE]]` (and `ELECTROMAGNETISM`/`THERMODYNAMICS` coverage). |